## Agent

In [ ]:
import random
from agents import Agent as AimaAgent, Thing as AimaThing, Direction

class Bump(AimaThing):
    def __init__(self, where): # 'forward' | 'left' | 'right'
        self.where = where

class Visited(AimaThing):
    def __init__(self, where):
        self.where = where

class Food(AimaThing):
    pass

class Water(AimaThing):
    pass

class SmartBlindDog(AimaAgent):

    visited: set[tuple[int, ...]]
    location: tuple[int, int]

    def __init__(self, program):
        super().__init__(program)
        self.visited = set()  
        self.location = (0, 0)
        self.direction = Direction("down")
        self.visited.add(tuple(self.location))

    def moveforward(self, success=True):
        if not success: return

        self.location = self.direction.move_forward(self.location)
        self.visited.add(tuple(self.location))

    def turn(self, d):
        self.direction = self.direction + d

    def eat(self, thing):
        ok = isinstance(thing, Food)
        if ok: self.performance += 10   # ให้คะแนนเมื่อกินสำเร็จ
        return ok

    def drink(self, thing):
        ok = isinstance(thing, Water)
        if ok: self.performance += 10   # ให้คะแนนเมื่อดื่มสำเร็จ
        return ok


def smart_program(percepts):
 
    blocked, walls = set(), set() # 'forward' | 'left' | 'right'

    for p in percepts:

        if isinstance(p, Food): return 'eat'

        elif isinstance(p, Water): return 'drink'

        elif isinstance(p, Bump):
            walls.add(p.where)
            blocked.add(p.where)

        elif isinstance(p, Visited):
            blocked.add(p.where)

    #  ยังมีทิศทางที่ไม่เคยไปเลย  สำรวจพื้นที่ใหม่ก่อนเสมอ
    fresh = [d for d in ('forward', 'left', 'right') if d not in blocked]
    if fresh:
        if 'forward' in fresh:
            return 'moveforward'
        return 'turnleft' if 'left' in fresh else 'turnright'

    #  รอบตัวเคยไปหมดแล้ว แต่ยังเดินได้ (ไม่ใช่กำแพง) -> เดินต่อ
    non_wall = [d for d in ('forward', 'left', 'right') if d not in walls]
    if non_wall:
        if 'forward' in non_wall:
            return 'moveforward'
        return 'turnleft' if 'left' in non_wall else 'turnright'


    return 'turnleft'

## environment

In [ ]:
from overrides import override

from agents import GraphicEnvironment

class NoRepeatPark(GraphicEnvironment):
    @override
    def is_inbounds(self, location):
        x, y = location
        return 0 <= x < self.width and 0 <= y < self.height
    
    def percept(self, agent):

        things = self.list_things_at(agent.location)

        for where, heading in (
            ('forward', agent.direction),
            ('left', agent.direction + Direction.L),
            ('right', agent.direction + Direction.R)
        ):
            loc = heading.move_forward(agent.location)

            if not self.is_inbounds(loc): things.append(Bump(where))
            elif tuple(loc) in agent.visited: things.append(Visited(where))

        return things

    def execute_action(self, agent, action):

        if action == 'turnright':
            agent.turn(Direction.R)
        elif action == 'turnleft':
            agent.turn(Direction.L)
        elif action == 'moveforward':
            agent.moveforward()

        elif action == 'eat':
            items = self.list_things_at(agent.location, tclass=Food)
            if items and agent.eat(items[0]): self.delete_thing(items[0])

        elif action == 'drink':
            items = self.list_things_at(agent.location, tclass=Water)
            if items and agent.drink(items[0]): self.delete_thing(items[0])

    def is_done(self):
        no_edibles = not any(isinstance(t, Food) or isinstance(t, Water) for t in self.things)
        dead_agents = not any(a.is_alive() for a in self.agents)
        return dead_agents or no_edibles

### Simulation

In [ ]:
park = NoRepeatPark(5, 5, color={'SmartBlindDog': (200,0,0), 'Water': (0,200,200), 'Food': (230,115,40)})
dog = SmartBlindDog(smart_program)
park.add_thing(dog, [0, 0])
park.add_thing(Food(), [3, 2])
park.add_thing(Water(), [2, 1])
park.run(100)

remaining = [t for t in park.things if isinstance(t, (Food, Water))]
print('เก็บของกินของดื่มครบหรือไม่:', 'ครบแล้ว!' if not remaining else f'ยังเหลือ {remaining}')
print('performance:', dog.performance)